# Andrey Baklykov

**Cluster 0 Stacking Model - Version 02**

**Personal project notebook** | Cluster 0: 525 companies, 3 bankrupt (0.57% rate)

This is the clean replay notebook for subgroup 0. All model-selection work was already completed in the sandbox notebook. Here we only reuse the finalized decisions, train the final subgroup model on the original train data, and save the `joblib` bundle for later testing.

In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.base import clone
from sklearn.decomposition import PCA
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler

from sklearn.model_selection import StratifiedKFold

SEED = 42
SUBGROUP_ID = 0
STUDENT_NAME = 'Baklykov'

PROJECT_DIR = Path.cwd().resolve()
while not (PROJECT_DIR / 'data').exists() and PROJECT_DIR.parent != PROJECT_DIR:
    PROJECT_DIR = PROJECT_DIR.parent

DATA_PATH = PROJECT_DIR / 'data' / 'clusters' / 'cluster_0.csv'
MODEL_OUTPUT_DIR = PROJECT_DIR / 'models' / 'subgroup_models'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
BUNDLE_PATH = MODEL_OUTPUT_DIR / 'cluster_0_model.joblib'

print(f'Project dir : {PROJECT_DIR}')
print(f'Data path   : {DATA_PATH}')
print(f'Bundle path : {BUNDLE_PATH}')


Project dir : S:\OneDriveMoleff\OneDrive - moleff.com\Documents\___STEVENS\CS_Program\CS559\HWs\CS-559-Project\CS-559-Project
Data path   : S:\OneDriveMoleff\OneDrive - moleff.com\Documents\___STEVENS\CS_Program\CS559\HWs\CS-559-Project\CS-559-Project\data\clusters\cluster_0.csv
Bundle path : S:\OneDriveMoleff\OneDrive - moleff.com\Documents\___STEVENS\CS_Program\CS559\HWs\CS-559-Project\CS-559-Project\models\subgroup_models\cluster_0_model.joblib


In [2]:
def compute_project_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    ff = int(((y_true == 0) & (y_pred == 0)).sum())
    ft = int(((y_true == 0) & (y_pred == 1)).sum())
    tt = int(((y_true == 1) & (y_pred == 1)).sum())
    tf = int(((y_true == 1) & (y_pred == 0)).sum())

    acc_eq1 = tt / (tt + tf) if (tt + tf) > 0 else np.nan
    pred_positive_rate = float(y_pred.mean())
    precision_positive = tt / (tt + ft) if (tt + ft) > 0 else np.nan

    return {
        'FF': ff,
        'FT': ft,
        'TT': tt,
        'TF': tf,
        'acc_eq1': acc_eq1,
        'pred_positive_rate': pred_positive_rate,
        'precision_positive': precision_positive,
    }

def safe_predict_proba(model, X):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, 'decision_function'):
        scores = np.asarray(model.decision_function(X), dtype=float)
        if scores.max() == scores.min():
            return np.full_like(scores, 0.5, dtype=float)
        return (scores - scores.min()) / (scores.max() - scores.min())
    raise ValueError(f"Model {type(model).__name__} has neither predict_proba nor decision_function.")

def build_engineered_meta_features(base_prob_df, passthrough_X=None):
    meta_df = base_prob_df.copy()

    plr = meta_df['lr_balanced'].values
    prf = meta_df['rf_balanced'].values
    pet = meta_df['et_balanced'].values

    meta_df['p_max'] = np.max(np.column_stack([plr, prf, pet]), axis=1)
    meta_df['p_mean'] = np.mean(np.column_stack([plr, prf, pet]), axis=1)
    meta_df['p_std'] = np.std(np.column_stack([plr, prf, pet]), axis=1)
    meta_df['p_rf_et_gap'] = np.abs(prf - pet)
    meta_df['p_lr_rf_gap'] = np.abs(plr - prf)
    meta_df['p_lr_et_gap'] = np.abs(plr - pet)
    meta_df['rf_et_both_gt_05'] = ((prf > 0.5) & (pet > 0.5)).astype(int)
    meta_df['rf_et_both_gt_03'] = ((prf > 0.3) & (pet > 0.3)).astype(int)
    meta_df['all_three_gt_03'] = ((plr > 0.3) & (prf > 0.3) & (pet > 0.3)).astype(int)

    if passthrough_X is not None:
        passthrough_df = pd.DataFrame(
            passthrough_X,
            index=meta_df.index,
            columns=[f'pass_{i}' for i in range(passthrough_X.shape[1])]
        )
        meta_df = pd.concat([meta_df, passthrough_df], axis=1)

    return meta_df

def get_oof_base_probabilities(X_raw, y, preprocessing, base_model_library, base_model_names, outer_cv, use_passthrough=True):
    n = X_raw.shape[0]
    oof_prob_df = pd.DataFrame(index=X_raw.index, columns=base_model_names, dtype=float)
    transformed_oof = None

    for train_idx, valid_idx in outer_cv.split(X_raw, y):
        X_train_raw = X_raw.iloc[train_idx]
        X_valid_raw = X_raw.iloc[valid_idx]
        y_train = y.iloc[train_idx]

        preprocess_fold = clone(preprocessing)
        X_train_trans = preprocess_fold.fit_transform(X_train_raw)
        X_valid_trans = preprocess_fold.transform(X_valid_raw)

        if use_passthrough:
            if transformed_oof is None:
                transformed_oof = np.zeros((n, X_valid_trans.shape[1]), dtype=float)
            transformed_oof[valid_idx, :] = X_valid_trans

        for base_name in base_model_names:
            base_model = clone(base_model_library[base_name])
            base_model.fit(X_train_trans, y_train)
            valid_prob = safe_predict_proba(base_model, X_valid_trans)
            oof_prob_df.loc[X_valid_raw.index, base_name] = valid_prob

    return oof_prob_df.astype(float), transformed_oof

def fit_base_models_on_full_train(X_raw, y, preprocessing, base_model_library, base_model_names):
    preprocess_fitted = clone(preprocessing)
    X_trans = preprocess_fitted.fit_transform(X_raw)

    fitted_base_models = {}
    full_prob_df = pd.DataFrame(index=X_raw.index, columns=base_model_names, dtype=float)

    for base_name in base_model_names:
        model = clone(base_model_library[base_name])
        model.fit(X_trans, y)
        fitted_base_models[base_name] = model
        full_prob_df[base_name] = safe_predict_proba(model, X_trans)

    return preprocess_fitted, X_trans, fitted_base_models, full_prob_df.astype(float)


---
## Load & Inspect Data

In [3]:
df0 = pd.read_csv(DATA_PATH)
index_col = df0['Index'] if 'Index' in df0.columns else pd.Series(range(len(df0)), name='Index')
y0 = df0['Bankrupt?'].astype(int).copy()

drop_cols = [c for c in ['Index', 'Bankrupt?'] if c in df0.columns]
X0_raw = df0.drop(columns=drop_cols).copy()

print(f'Cluster 0: {len(df0)} samples, {X0_raw.shape[1]} raw features')
print(f'Bankrupt: {int(y0.sum())} ({y0.mean():.4%})')
print(f'Non-bankrupt: {int((y0 == 0).sum())}')

data_summary_df = pd.DataFrame([{
    'subgroup_id': SUBGROUP_ID,
    'student_name': STUDENT_NAME,
    'rows': int(len(df0)),
    'raw_feature_count': int(X0_raw.shape[1]),
    'bankrupt': int(y0.sum()),
    'non_bankrupt': int((y0 == 0).sum()),
    'bankruptcy_rate': float(y0.mean()),
}])
display(data_summary_df)


Cluster 0: 525 samples, 95 raw features
Bankrupt: 3 (0.5714%)
Non-bankrupt: 522


,subgroup_id,student_name,rows,raw_feature_count,bankrupt,non_bankrupt,bankruptcy_rate
0,0,Baklykov,525,95,3,522,0.005714


## Final Feature Set

In [4]:
selected_features = [
    'ROA(C) before interest and depreciation before interest',
    'Operating Gross Margin',
    'Operating Profit Rate',
    'Working Capital to Total Assets',
    'Cash/Total Assets',
    'Cash/Current Liability',
    'Current Liability to Assets',
    'Operating Funds to Liability',
    'Cash Flow to Total Assets',
    'Cash Flow to Liability',
    'Current Ratio',
    'Quick Ratio',
    'Debt ratio %',
    'Total debt/Total net worth',
    'Borrowing dependency',
    'Contingent liabilities/Net worth',
    'Interest Coverage Ratio (Interest expense to EBIT)',
    'Interest Expense Ratio',
    'Retained Earnings to Total Assets',
    'Equity to Liability'
]

X0_selected = X0_raw[selected_features].copy()
Nfeatures = len(selected_features)

print(f'Selected feature count: {Nfeatures}')
selected_features_df = pd.DataFrame({
    'feature_number': range(1, Nfeatures + 1),
    'feature_name': selected_features,
})
display(selected_features_df)


Selected feature count: 20


,feature_number,feature_name
0,1,ROA(C) before interest and depreciation before...
1,2,Operating Gross Margin
2,3,Operating Profit Rate
3,4,Working Capital to Total Assets
4,5,Cash/Total Assets
5,6,Cash/Current Liability
6,7,Current Liability to Assets
7,8,Operating Funds to Liability
8,9,Cash Flow to Total Assets
9,10,Cash Flow to Liability


## Final Fixed Configuration

In [5]:
TARGET_PREPROCESSING_NAME = 'power_pca_8'
TARGET_BASE_MODEL_NAMES = ['lr_balanced', 'rf_balanced', 'et_balanced']
TARGET_META_MODEL_NAME = 'meta_rf_shallow_plain_d3_l5'
TARGET_BEST_THRESHOLD = 0.16
USE_PASSTHROUGH = True

TARGET_PREPROCESSING_TEMPLATE = Pipeline([
    ('power', PowerTransformer()),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=min(8, Nfeatures), random_state=SEED))
])

BASE_MODEL_LIBRARY = {
    'lr_balanced': LogisticRegression(
        max_iter=3000,
        class_weight='balanced',
        solver='liblinear',
        random_state=SEED
    ),
    'rf_balanced': RandomForestClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    ),
    'et_balanced': ExtraTreesClassifier(
        n_estimators=400,
        max_depth=None,
        min_samples_leaf=2,
        class_weight='balanced_subsample',
        random_state=SEED,
        n_jobs=-1
    )
}

TARGET_META_MODEL_TEMPLATE = RandomForestClassifier(
    n_estimators=300,
    max_depth=3,
    min_samples_leaf=5,
    class_weight=None,
    random_state=SEED,
    n_jobs=-1
)

loaded_config_df = pd.DataFrame([{
    'preprocessing_name': TARGET_PREPROCESSING_NAME,
    'base_models': ', '.join(TARGET_BASE_MODEL_NAMES),
    'selected_meta_model': TARGET_META_MODEL_NAME,
    'threshold': TARGET_BEST_THRESHOLD,
    'feature_count': Nfeatures,
    'use_passthrough': USE_PASSTHROUGH,
}])
display(loaded_config_df)


,preprocessing_name,base_models,selected_meta_model,threshold,feature_count,use_passthrough
0,power_pca_8,"lr_balanced, rf_balanced, et_balanced",meta_rf_shallow_plain_d3_l5,0.16,20,True


## Cross-Validation

In [6]:
X_subgroup = X0_selected.copy()
y_subgroup = y0.copy()

n_bankrupt = int(y_subgroup.sum())
n_non_bankrupt = int((y_subgroup == 0).sum())
min_class_count = int(min(n_bankrupt, n_non_bankrupt))
outer_cv_splits = min(3, min_class_count)
outer_cv = StratifiedKFold(n_splits=outer_cv_splits, shuffle=True, random_state=SEED)

base_oof_prob_df, passthrough_oof_X = get_oof_base_probabilities(
    X_raw=X_subgroup,
    y=y_subgroup,
    preprocessing=clone(TARGET_PREPROCESSING_TEMPLATE),
    base_model_library=BASE_MODEL_LIBRARY,
    base_model_names=TARGET_BASE_MODEL_NAMES,
    outer_cv=outer_cv,
    use_passthrough=USE_PASSTHROUGH
)

meta_oof_X = build_engineered_meta_features(base_oof_prob_df, passthrough_X=passthrough_oof_X if USE_PASSTHROUGH else None)
OOF_META_FEATURE_COLUMNS = meta_oof_X.columns.tolist()

oof_meta_model = clone(TARGET_META_MODEL_TEMPLATE)
oof_meta_model.fit(meta_oof_X, y_subgroup)

target_oof_prob = safe_predict_proba(oof_meta_model, meta_oof_X)
target_oof_pred = (target_oof_prob >= TARGET_BEST_THRESHOLD).astype(int)
target_oof_metrics = compute_project_metrics(y_subgroup.values, target_oof_pred)

oof_result_df = pd.DataFrame([{
    'dataset': 'oof_cross_validation',
    'preprocessing_name': TARGET_PREPROCESSING_NAME,
    'base_models': ', '.join(TARGET_BASE_MODEL_NAMES),
    'meta_model_name': TARGET_META_MODEL_NAME,
    'threshold': TARGET_BEST_THRESHOLD,
    'meta_feature_count': meta_oof_X.shape[1],
    **target_oof_metrics,
}])

display(oof_result_df)


,dataset,preprocessing_name,base_models,meta_model_name,threshold,meta_feature_count,FF,FT,TT,TF,acc_eq1,pred_positive_rate,precision_positive
0,oof_cross_validation,power_pca_8,"lr_balanced, rf_balanced, et_balanced",meta_rf_shallow_plain_d3_l5,0.16,20,520,2,3,0,1.0,0.009524,0.6


## Train Final Model

In [7]:
X_subgroup = X0_selected.copy()
y_subgroup = y0.copy()

final_preprocess_fitted, final_X_trans, final_base_models, final_base_prob_df = fit_base_models_on_full_train(
    X_raw=X_subgroup,
    y=y_subgroup,
    preprocessing=clone(TARGET_PREPROCESSING_TEMPLATE),
    base_model_library=BASE_MODEL_LIBRARY,
    base_model_names=TARGET_BASE_MODEL_NAMES
)

final_meta_X = build_engineered_meta_features(final_base_prob_df, passthrough_X=final_X_trans if USE_PASSTHROUGH else None)
META_FEATURE_COLUMNS = final_meta_X.columns.tolist()

final_meta_model = clone(TARGET_META_MODEL_TEMPLATE)
final_meta_model.fit(final_meta_X, y_subgroup)

target_train_prob = safe_predict_proba(final_meta_model, final_meta_X)
target_train_pred = (target_train_prob >= TARGET_BEST_THRESHOLD).astype(int)
target_train_metrics = compute_project_metrics(y_subgroup.values, target_train_pred)

train_result_df = pd.DataFrame([{
    'dataset': 'original_subgroup_train',
    'preprocessing_name': TARGET_PREPROCESSING_NAME,
    'base_models': ', '.join(TARGET_BASE_MODEL_NAMES),
    'meta_model_name': TARGET_META_MODEL_NAME,
    'threshold': TARGET_BEST_THRESHOLD,
    'meta_feature_count': final_meta_X.shape[1],
    **target_train_metrics,
}])

table3_row_df = pd.DataFrame([{
    'Subgroup ID': SUBGROUP_ID,
    'Name of Student': STUDENT_NAME,
    'Number of Companies': int(len(y_subgroup)),
    'Number Bankrupt': int(y_subgroup.sum()),
    'TT': int(target_train_metrics['TT']),
    'TF': int(target_train_metrics['TF']),
    'Nfeatures': int(Nfeatures),
    'acc': float(target_train_metrics['acc_eq1']),
}])

comparison_df = pd.DataFrame([
    {
        'split': 'Cross-Validation',
        'TT': target_oof_metrics['TT'],
        'TF': target_oof_metrics['TF'],
        'FT': target_oof_metrics['FT'],
        'acc_eq1': target_oof_metrics['acc_eq1'],
        'pred_positive_rate': target_oof_metrics['pred_positive_rate'],
        'precision_positive': target_oof_metrics['precision_positive'],
    },
    {
        'split': 'Training',
        'TT': target_train_metrics['TT'],
        'TF': target_train_metrics['TF'],
        'FT': target_train_metrics['FT'],
        'acc_eq1': target_train_metrics['acc_eq1'],
        'pred_positive_rate': target_train_metrics['pred_positive_rate'],
        'precision_positive': target_train_metrics['precision_positive'],
    },
])

display(train_result_df)
display(table3_row_df)
display(comparison_df)


,dataset,preprocessing_name,base_models,meta_model_name,threshold,meta_feature_count,FF,FT,TT,TF,acc_eq1,pred_positive_rate,precision_positive
0,original_subgroup_train,power_pca_8,"lr_balanced, rf_balanced, et_balanced",meta_rf_shallow_plain_d3_l5,0.16,20,518,4,3,0,1.0,0.013333,0.428571


,Subgroup ID,Name of Student,Number of Companies,Number Bankrupt,TT,TF,Nfeatures,acc
0,0,Baklykov,525,3,3,0,20,1.0


,split,TT,TF,FT,acc_eq1,pred_positive_rate,precision_positive
0,Cross-Validation,3,0,2,1.0,0.009524,0.600000
1,Training,3,0,4,1.0,0.013333,0.428571


## Save Model Bundle

In [8]:
class Cluster0StackingWrapper:
    """Wraps Andrey's multi-step pipeline into the standard predict_proba interface."""
    def __init__(self, preprocess_fitted, fitted_base_models, base_model_names,
                 fitted_meta_model, use_passthrough, feature_names):
        self.preprocess_fitted = preprocess_fitted
        self.fitted_base_models = fitted_base_models
        self.base_model_names = base_model_names
        self.fitted_meta_model = fitted_meta_model
        self.use_passthrough = use_passthrough
        self.feature_names = feature_names

    def predict_proba(self, X):
        if not isinstance(X, pd.DataFrame):
            X = pd.DataFrame(X, columns=self.feature_names)
        X_trans = self.preprocess_fitted.transform(X)
        base_prob_df = pd.DataFrame(
            {name: safe_predict_proba(self.fitted_base_models[name], X_trans)
             for name in self.base_model_names}
        )
        meta_X = build_engineered_meta_features(
            base_prob_df,
            passthrough_X=X_trans if self.use_passthrough else None
        )
        proba_pos = safe_predict_proba(self.fitted_meta_model, meta_X)
        return np.column_stack([1 - proba_pos, proba_pos])


stacker_wrapper = Cluster0StackingWrapper(
    preprocess_fitted=final_preprocess_fitted,
    fitted_base_models=final_base_models,
    base_model_names=TARGET_BASE_MODEL_NAMES,
    fitted_meta_model=final_meta_model,
    use_passthrough=USE_PASSTHROUGH,
    feature_names=selected_features
)

final_bundle = {
    # Standard keys - compatible with Group2_Generalization.ipynb
    'model': stacker_wrapper,
    'features': selected_features,
    'n_features': int(Nfeatures),
    'threshold': float(TARGET_BEST_THRESHOLD),
    'cluster_id': SUBGROUP_ID,
    'model_type': 'stacking',
    'member': STUDENT_NAME,
    # Andrey's additional metadata (kept for reference so it doesnt mess any of his stuff up)
    'subgroup_id': SUBGROUP_ID,
    'student_name': STUDENT_NAME,
    'selected_features': selected_features,
    'preprocessing_name': TARGET_PREPROCESSING_NAME,
    'base_model_names': TARGET_BASE_MODEL_NAMES,
    'meta_model_name': TARGET_META_MODEL_NAME,
    'use_passthrough': USE_PASSTHROUGH,
    'meta_feature_columns': META_FEATURE_COLUMNS,
    'preprocess_fitted': final_preprocess_fitted,
    'fitted_base_models': final_base_models,
    'fitted_meta_model': final_meta_model,
    'oof_metrics': target_oof_metrics,
    'train_metrics': target_train_metrics,
    'table3_metrics': table3_row_df.iloc[0].to_dict(),
}

joblib.dump(final_bundle, BUNDLE_PATH)

bundle_overview_df = pd.DataFrame({
    'bundle_key': list(final_bundle.keys())
})

print(f'Saved bundle: {BUNDLE_PATH}')
display(bundle_overview_df)


Saved bundle: S:\OneDriveMoleff\OneDrive - moleff.com\Documents\___STEVENS\CS_Program\CS559\HWs\CS-559-Project\CS-559-Project\models\subgroup_models\cluster_0_model.joblib


,bundle_key
0,model
1,features
2,n_features
3,threshold
4,cluster_id
5,model_type
6,member
7,subgroup_id
8,student_name
9,selected_features


## Summary Cluster 0

| Metric | Cross-Validation | Training |
|---|---|---|
| Custom Accuracy (TT/(TT+TF)) | 100.00% | 100.00% |
| TT (bankrupt correctly predicted) | 3 | 3 |
| TF (bankrupt missed) | 0 | 0 |
| FT (non-bankrupt flagged as bankrupt) | 2 | 4 |
| Predicted Positive Rate | 0.95% | 1.33% |
| Precision Positive | 60.00% | 42.86% |
| Features (N) | 20 | 20 |
| Threshold | 0.16 | 0.16 |

**Selected preprocessing:** `power_pca_8`

**Base models:** `lr_balanced`, `rf_balanced`, `et_balanced`

**Selected meta-model:** `meta_rf_shallow_plain_d3_l5`

**Saved bundle:** `models/subgroup_models/subgroup_0_stacking_bundle_version_02.joblib`

**Interpretation:** This notebook does not repeat model selection. It only replays the finalized subgroup-0 solution, fits it on the original train data, and exports the bundle for later test-time use.